# 18.9 Data versioning and lineage

Data versioning makes model behavior reproducible by remembering exactly which data existed when. Content hashes, diffs, transform parameters, and model-to-data indexes turn debugging and rollback into graph queries. Save a copy to Drive to edit.

In [ ]:

import hashlib
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(18)


def dataquality_ladder():
    """Breast Cancer ladder with progressively degraded data quality."""
    bc = load_breast_cancer()
    X0 = bc.data.astype(float)
    y0 = bc.target
    rng = np.random.default_rng(18)
    rungs = []

    rungs.append(("D1 clean", X0.copy(), y0.copy()))

    y2 = y0.copy()
    flip = rng.random(y2.shape) < 0.15
    y2[flip] = 1 - y2[flip]
    rungs.append(("D2 15% label noise", X0.copy(), y2))

    keep_pos = np.where(y0 == 1)[0]
    keep_neg = np.where(y0 == 0)[0][:30]
    idx = np.concatenate([keep_pos, keep_neg])
    rungs.append(("D3 class imbalance", X0[idx].copy(), y0[idx].copy()))

    X4 = X0 + rng.normal(0.0, X0.std(axis=0) * 0.5, size=X0.shape)
    rungs.append(("D4 heavy feature noise", X4, y0.copy()))

    X5 = X0.copy()
    holes = rng.random(X5.shape) < 0.2
    X5[holes] = np.nan
    col_means = np.nanmean(X5, axis=0)
    X5 = np.where(np.isnan(X5), col_means, X5)
    rungs.append(("D5 20% missing (mean-imputed)", X5, y0.copy()))

    return rungs


def stable_split(X, y):
    return train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=18,
        stratify=y,
    )


def fit_logistic_accuracy(x_train, y_train, x_test, y_test):
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_train_scaled, y_train)
    pred = clf.predict(x_test_scaled)
    return float(accuracy_score(y_test, pred))


def preview_ladder(rungs):
    for name, X, y in rungs:
        counts = np.bincount(y.astype(int))
        sample = np.round(X[:2, :4], 3)
        print(name)
        print("  shape", X.shape, "class_counts", counts.tolist())
        print("  first_two_by_four")
        print(sample)


## The concept, built once (D1)

A dataset version is identified by a content hash $h(D)$, a diff $D_{t+1}\setminus D_t$, and lineage edges for transforms. The lesson checks old $\{1,2,3,4\}$ vs new $\{1,2,4,5\}$, lineage counts $3\to2\to2$, and rollback to the previous passing version 2.

In [ ]:

def snapshot_hash(rows):
    payload = ",".join(str(int(v)) for v in sorted(rows))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]


old_rows = {1, 2, 3, 4}
new_rows = {1, 2, 4, 5}
added = sorted(new_rows - old_rows)
removed = sorted(old_rows - new_rows)
kept = sorted(old_rows & new_rows)
raw = np.array([1, 2, 3])
cleaned = raw[raw >= 2]
featured = cleaned ** 2
scores = np.array([0.91, 0.93, 0.62, 0.94])
threshold = 0.90
failed_version = int(np.where(scores < threshold)[0][0] + 1)
previous_passing = int(np.where(scores[:failed_version - 1] >= threshold)[0][-1] + 1)
print("old hash", snapshot_hash(old_rows), "new hash", snapshot_hash(new_rows))
print("added", added, "removed", removed, "kept", kept)
print("lineage counts", len(raw), len(cleaned), len(featured))
print("failed version", failed_version, "rollback", previous_passing)
assert len(added) == 1
assert len(removed) == 1
assert len(kept) == 3
assert [len(raw), len(cleaned), len(featured)] == [3, 2, 2]
assert failed_version == 3
assert previous_passing == 2


Now train from a selected snapshot. The method records hashes, row diffs, cleaning parameters, and a model-to-data pointer before fitting the same classifier.

In [ ]:

def array_hash(X, y):
    X_round = np.nan_to_num(np.round(X, 6), nan=-999999.0)
    payload = X_round.tobytes() + y.astype(int).tobytes()
    return hashlib.sha256(payload).hexdigest()[:12]


def versioned_train(X, y):
    rng = np.random.default_rng(189)
    versions = []
    versions.append({"version": 1, "X": X.copy(), "y": y.copy(), "quality": 0.92, "transform": "raw"})
    y_bad = y.copy()
    flip = rng.choice(len(y_bad), size=max(1, len(y_bad) // 8), replace=False)
    y_bad[flip] = 1 - y_bad[flip]
    versions.append({"version": 2, "X": X.copy(), "y": y_bad, "quality": 0.88, "transform": "label_noise"})
    imputer = SimpleImputer(strategy="median")
    X_clean = imputer.fit_transform(X)
    versions.append({"version": 3, "X": X_clean, "y": y.copy(), "quality": 0.94, "transform": "median_impute"})
    passing = [v for v in versions if v["quality"] >= 0.90]
    selected = passing[-1]
    x_train, x_test, y_train, y_test = stable_split(selected["X"], selected["y"])
    accuracy = fit_logistic_accuracy(x_train, y_train, x_test, y_test)
    lineage = {
        "selected_version": selected["version"],
        "selected_hash": array_hash(selected["X"], selected["y"]),
        "transform": selected["transform"],
        "median_parameters": imputer.statistics_.tolist(),
        "model_to_data": {"logistic_model": selected["version"]},
        "artifact": selected["X"][:80, :2],
    }
    return float(accuracy), lineage


## The dataset ladder

In [ ]:

rungs = dataquality_ladder()
preview_ladder(rungs)


## Run the same method across D1-D5

In [ ]:

results = []
for name, X, y in rungs:
    accuracy, lineage = versioned_train(X, y)
    row = {
        "rung": name,
        "accuracy": accuracy,
        "version": lineage["selected_version"],
        "hash": lineage["selected_hash"],
        "artifact": lineage["artifact"],
    }
    results.append(row)
    print(f"{name:28s} acc={row['accuracy']:.3f} version={row['version']} hash={row['hash']}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, row in zip(axes, results):
    art = row["artifact"]
    ax.scatter(art[:, 0], art[:, 1], s=10, alpha=0.7)
    ax.set_title(row["rung"].split()[0] + " v" + str(row["version"]))
    ax.set_xlabel("snapshot f0")
    ax.set_ylabel("snapshot f1")
fig.suptitle("Selected version artifacts")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), [r["accuracy"] for r in results], marker="o")
plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0.6, 1.0)
plt.ylabel("snapshot-selected accuracy")
plt.title("Versioned training by data quality")
plt.grid(True)
plt.show()


## Pitfall on D5: losing transform parameters

A cleaned snapshot without the imputer values cannot be reproduced. The fix stores transform parameters and the downstream model-to-data index.

In [ ]:

name, X, y = rungs[-1]
rng = np.random.default_rng(918)
raw_with_holes = X.copy()
holes = rng.random(raw_with_holes.shape) < 0.10
raw_with_holes[holes] = np.nan
bad_imputer = SimpleImputer(strategy="mean")
bad_clean = bad_imputer.fit_transform(raw_with_holes[: len(raw_with_holes) // 2])
repeat_imputer = SimpleImputer(strategy="mean")
repeat_clean = repeat_imputer.fit_transform(raw_with_holes)
bad_hash = array_hash(bad_clean, y[: len(bad_clean)])
repeat_hash = array_hash(repeat_clean, y)
_, fixed_lineage = versioned_train(raw_with_holes, y)
print("untracked partial hash", bad_hash)
print("untracked full hash", repeat_hash)
print("stored transform", fixed_lineage["transform"])
print("model index", fixed_lineage["model_to_data"])
assert bad_hash != repeat_hash
assert fixed_lineage["model_to_data"]["logistic_model"] == fixed_lineage["selected_version"]
assert len(fixed_lineage["median_parameters"]) == X.shape[1]



## Evaluate it + Practice

- Compare the reported accuracy to a no-skill majority-class baseline for every rung.
- Sanity-check that the key data artifact changes in the intended direction before trusting the metric.
- Ablate the key data-centric step, such as filtering, augmentation, validation, version selection, or valuation-guided dropping.
- Watch for failure signals: gains only on corrupted validation data, impossible feature values, leakage, or unstable row rankings.
- Re-run with a new seed and require the conclusion, not every decimal, to remain stable.

Practice 1: change the corruption or repair strength and re-run the D1 to D5 curve.


Practice 2: add one slice-level diagnostic for the hardest rung.

Practice 3: replace accuracy with balanced accuracy or F1 and compare the story.